In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
baslangic_zamani = time.time()

spark = SparkSession.builder \
    .appName("PySpark_Baseline") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


ulkeler_ve_dosyalar = {
    "Türkiye": "Datas/turkiye.csv",
    "Romanya": "Datas/romanya.csv",
    "Libya": "Datas/libya.csv",
    "KKTC": "Datas/kktc.csv",
    "Azerbaycan": "Datas/azerbaycan.csv"
}

df_global = None

for ulke, dosya_yolu in ulkeler_ve_dosyalar.items():
    df_temp = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "iso-8859-1") \
    .csv(dosya_yolu, header=True, inferSchema=True)
    
    if df_global is None:
        df_global = df_temp
    else:
        df_global = df_global.unionByName(df_temp, allowMissingColumns=True)


toplam_satir = df_global.count()

bitis_zamani = time.time()
gecen_sure = bitis_zamani - baslangic_zamani

print("-" * 50)
print(f"Toplam Satır Sayısı: {toplam_satir}")
print(f"PySpark İşlem Süresi: {gecen_sure:.3f} saniye")
print("-" * 50)

df_global.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 02:05:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


--------------------------------------------------
Toplam Satır Sayısı: 1077380
PySpark İşlem Süresi: 18.808 saniye
--------------------------------------------------


26/04/27 02:05:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+----------+-------------+--------+-------+--------------------+---------+--------------------+---------------+--------------+--------------------+---------+----------------+------------+---------------------+----------------------+----------------------+-----------+-----------------+------------------------+--------+-----------------+------------------+-----------------+----------------+------+------+-----------+-----------+
|merchyil|merchyilay|merchyilhafta|yilaygun|   ulke|              magaza|magazakod|merchmarkayasgrupkod|merchaltgrupkod|urunklasmankod|    urunklasmantanim|satisadet|satisadet_lfl_gy|indirimorani|gunlukminimumsicaklik|gunlukortalamasicaklik|gunlukmaksimumsicaklik|ozelgunflag|  depoyerlesimtip|magazamusterigirissayisi|magazam2|magazaacilissaati|magazakapanissaati|gunsonutoplamstok|gunsonureyonstok|ilkpsf|sonpsf|total_cover|reyon_cover|
+--------+----------+-------------+--------+-------+--------------------+---------+--------------------+---------------+--

In [3]:
df_global.printSchema()
tarama_ifadeleri = []
for sutun_adi, sutun_tipi in df_global.dtypes:
    
    if sutun_tipi in ['double', 'float']:
        tarama_ifadeleri.append(F.count(F.when(F.isnan(sutun_adi) | F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))
    else:
        
        tarama_ifadeleri.append(F.count(F.when(F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))
df_global.select(*tarama_ifadeleri).show()


print("TEMEL İSTATİSTİKLER:")
df_global.describe().show()

root
 |-- merchyil: integer (nullable = true)
 |-- merchyilay: integer (nullable = true)
 |-- merchyilhafta: integer (nullable = true)
 |-- yilaygun: integer (nullable = true)
 |-- ulke: string (nullable = true)
 |-- magaza: string (nullable = true)
 |-- magazakod: string (nullable = true)
 |-- merchmarkayasgrupkod: string (nullable = true)
 |-- merchaltgrupkod: string (nullable = true)
 |-- urunklasmankod: integer (nullable = true)
 |-- urunklasmantanim: string (nullable = true)
 |-- satisadet: integer (nullable = true)
 |-- satisadet_lfl_gy: double (nullable = true)
 |-- indirimorani: double (nullable = true)
 |-- gunlukminimumsicaklik: double (nullable = true)
 |-- gunlukortalamasicaklik: double (nullable = true)
 |-- gunlukmaksimumsicaklik: double (nullable = true)
 |-- ozelgunflag: integer (nullable = true)
 |-- depoyerlesimtip: string (nullable = true)
 |-- magazamusterigirissayisi: long (nullable = true)
 |-- magazam2: double (nullable = true)
 |-- magazaacilissaati: string (nul

+--------+----------+-------------+--------+----+------+---------+--------------------+---------------+--------------+----------------+---------+----------------+------------+---------------------+----------------------+----------------------+-----------+---------------+------------------------+--------+-----------------+------------------+-----------------+----------------+------+------+-----------+-----------+
|merchyil|merchyilay|merchyilhafta|yilaygun|ulke|magaza|magazakod|merchmarkayasgrupkod|merchaltgrupkod|urunklasmankod|urunklasmantanim|satisadet|satisadet_lfl_gy|indirimorani|gunlukminimumsicaklik|gunlukortalamasicaklik|gunlukmaksimumsicaklik|ozelgunflag|depoyerlesimtip|magazamusterigirissayisi|magazam2|magazaacilissaati|magazakapanissaati|gunsonutoplamstok|gunsonureyonstok|ilkpsf|sonpsf|total_cover|reyon_cover|
+--------+----------+-------------+--------+----+------+---------+--------------------+---------------+--------------+----------------+---------+----------------+------

+-------+------------------+----------------+------------------+--------------------+----------+---------------+---------+--------------------+---------------+------------------+-------------------+------------------+------------------+-------------------+---------------------+----------------------+----------------------+-------------------+---------------+------------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-----------------+-----------------+
|summary|          merchyil|      merchyilay|     merchyilhafta|            yilaygun|      ulke|         magaza|magazakod|merchmarkayasgrupkod|merchaltgrupkod|    urunklasmankod|   urunklasmantanim|         satisadet|  satisadet_lfl_gy|       indirimorani|gunlukminimumsicaklik|gunlukortalamasicaklik|gunlukmaksimumsicaklik|        ozelgunflag|depoyerlesimtip|magazamusterigirissayisi|          magazam2|magazaacilissaati|magazakapanissaati|gunson

In [4]:
df_global = df_global.withColumn("Gercek_Tarih", F.to_date(F.col("yilaygun").cast("string"), "yyyyMMdd"))
df_global = df_global.withColumn("Haftanin_Gunu", F.dayofweek(F.col("Gercek_Tarih")))
df_global = df_global.withColumn("Yilin_Ayi", F.month(F.col("Gercek_Tarih")))   
df_global = df_global.withColumn("Yilin_Haftasi", F.weekofyear("Gercek_Tarih"))
df_global = df_global.withColumn("magazaacilissaati", F.try_to_timestamp(F.try_to_timestamp(F.col("magazaacilissaati"), F.lit("M/d/yyyy H:mm"))))
df_global = df_global.withColumn("magazakapanissaati", F.try_to_timestamp(F.try_to_timestamp(F.col("magazakapanissaati"), F.lit("M/d/yyyy H:mm"))))
df_global = df_global.withColumn("Acik_Kalma_Suresi_Saat",F.round((F.col("magazakapanissaati").cast("long") - F.col("magazaacilissaati").cast("long")) / 3600, 2))
df_global.select("yilaygun", "Gercek_Tarih", "Haftanin_Gunu", "magazaacilissaati", "magazakapanissaati", "Acik_Kalma_Suresi_Saat", "Yilin_Ayi", "Yilin_Haftasi").show(5)

+--------+------------+-------------+-------------------+-------------------+----------------------+---------+-------------+
|yilaygun|Gercek_Tarih|Haftanin_Gunu|  magazaacilissaati| magazakapanissaati|Acik_Kalma_Suresi_Saat|Yilin_Ayi|Yilin_Haftasi|
+--------+------------+-------------+-------------------+-------------------+----------------------+---------+-------------+
|20230101|  2023-01-01|            1|2023-01-01 10:00:00|2023-01-01 22:21:00|                 12.35|        1|           52|
|20230101|  2023-01-01|            1|2023-01-01 10:00:00|2023-01-01 22:21:00|                 12.35|        1|           52|
|20230101|  2023-01-01|            1|2023-01-01 10:00:00|2023-01-01 22:21:00|                 12.35|        1|           52|
|20230101|  2023-01-01|            1|2023-01-01 10:00:00|2023-01-01 22:21:00|                 12.35|        1|           52|
|20230101|  2023-01-01|            1|2023-01-01 10:00:00|2023-01-01 22:21:00|                 12.35|        1|           52|


In [5]:
tarama_ifadeleri = []
for sutun_adi, sutun_tipi in df_global.dtypes:
    if sutun_tipi in ['double', 'float', 'integer', 'long']:
        tarama_ifadeleri.append(F.count(F.when(F.isnan(sutun_adi) | F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))
    else:
        tarama_ifadeleri.append(F.count(F.when(F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))


df_eksik = df_global.select(*tarama_ifadeleri)


eksik_sozluk = df_eksik.collect()[0].asDict()
sorunlu_sutunlar = {k: v for k, v in eksik_sozluk.items() if v > 0}

print("-" * 50)
if len(sorunlu_sutunlar) == 0:
    print("Veri setinde hiç eksik (Null) değer yok!")
else:
    print("Aşağıdaki sütunlarda eksik veriler tespit edildi:")
    for sutun, adet in sorunlu_sutunlar.items():
        print(f" -> {sutun}: {adet} satır kayıp. Kayıp yüzdesi {adet/(df_global.count()):.2%}")
print("-" * 50)

--------------------------------------------------
Aşağıdaki sütunlarda eksik veriler tespit edildi:
 -> indirimorani: 897374 satır kayıp. Kayıp yüzdesi 83.29%
 -> magazamusterigirissayisi: 583616 satır kayıp. Kayıp yüzdesi 54.17%
 -> magazam2: 805112 satır kayıp. Kayıp yüzdesi 74.73%
 -> magazaacilissaati: 155815 satır kayıp. Kayıp yüzdesi 14.46%
 -> magazakapanissaati: 147151 satır kayıp. Kayıp yüzdesi 13.66%
 -> ilkpsf: 898181 satır kayıp. Kayıp yüzdesi 83.37%
 -> sonpsf: 898181 satır kayıp. Kayıp yüzdesi 83.37%
 -> total_cover: 1060521 satır kayıp. Kayıp yüzdesi 98.44%
 -> reyon_cover: 1060521 satır kayıp. Kayıp yüzdesi 98.44%
 -> Acik_Kalma_Suresi_Saat: 156025 satır kayıp. Kayıp yüzdesi 14.48%
--------------------------------------------------


In [6]:
from pyspark.ml.feature import Imputer
DROP_COLS = [
    "total_cover", "reyon_cover", 
    "ilkpsf", "sonpsf", 
    "magazam2", "magazamusterigirissayisi",
    "magazaacilissaati", "magazakapanissaati",
    "urunklasmantanim","merchyil","merchyilay","merchyilhafta","yilaygun","magaza","Gercek_Tarih"
]
df_clean = df_global.drop(*DROP_COLS)
df_clean = df_clean.fillna({"indirimorani": 0.0})
imputer = Imputer(
    inputCols=["Acik_Kalma_Suresi_Saat"],
    outputCols=["Acik_Kalma_Suresi_Saat"], 
    strategy="median"
)
df_clean = imputer.fit(df_clean).transform(df_clean)

print("Temizlik Tamamlandı!")

Temizlik Tamamlandı!


In [7]:
df_clean = df_clean.withColumn("Haftasonu_Mu", F.when(F.col("Haftanin_Gunu").isin(1, 7), 1).otherwise(0))
df_clean = df_clean.withColumn("Indirim_x_Haftasonu", F.col("indirimorani") * F.col("Haftasonu_Mu"))

In [8]:
CATEGORİCAL_COLS = [col for col, dtype in df_clean.dtypes if dtype == "string"]
#Cardinality kontrolü
print("\nKategorik Sütunların Kardinalitesi (Benzersiz Değer Sayısı):")
for sutun in CATEGORİCAL_COLS:
    benzersiz_sayisi = df_clean.select(sutun).distinct().count()
    print(f" -> {sutun}: {benzersiz_sayisi} benzersiz değer")


Kategorik Sütunların Kardinalitesi (Benzersiz Değer Sayısı):


 -> ulke: 5 benzersiz değer


 -> magazakod: 128 benzersiz değer
 -> merchmarkayasgrupkod: 5 benzersiz değer
 -> merchaltgrupkod: 6 benzersiz değer
 -> depoyerlesimtip: 5 benzersiz değer


In [9]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

print("Kategorik Veri Kodlama (Encoding)")

# 2. KODLANACAK SÜTUNLAR LİSTESİ
kategorik_sutunlar = [
    'ulke', 
    'magazakod', 
    'merchmarkayasgrupkod', 
    'merchaltgrupkod', 
    'depoyerlesimtip'
]

indexers = [
    StringIndexer(
        inputCol=sutun, 
        outputCol=sutun + "_Index", 
        handleInvalid="keep" 
    )
    for sutun in kategorik_sutunlar
]


pipeline_kategorik = Pipeline(stages=indexers)
df_ml_hazir = pipeline_kategorik.fit(df_clean).transform(df_clean)


df_ml_hazir = df_ml_hazir.drop(*kategorik_sutunlar)

print("Tüm kategorik metinler başarıyla matematiksel ID'lere dönüştürüldü!")
df_ml_hazir.printSchema()

Kategorik Veri Kodlama (Encoding)
Tüm kategorik metinler başarıyla matematiksel ID'lere dönüştürüldü!
root
 |-- urunklasmankod: integer (nullable = true)
 |-- satisadet: integer (nullable = true)
 |-- satisadet_lfl_gy: double (nullable = true)
 |-- indirimorani: double (nullable = false)
 |-- gunlukminimumsicaklik: double (nullable = true)
 |-- gunlukortalamasicaklik: double (nullable = true)
 |-- gunlukmaksimumsicaklik: double (nullable = true)
 |-- ozelgunflag: integer (nullable = true)
 |-- gunsonutoplamstok: integer (nullable = true)
 |-- gunsonureyonstok: integer (nullable = true)
 |-- Haftanin_Gunu: integer (nullable = true)
 |-- Yilin_Ayi: integer (nullable = true)
 |-- Yilin_Haftasi: integer (nullable = true)
 |-- Acik_Kalma_Suresi_Saat: double (nullable = true)
 |-- Haftasonu_Mu: integer (nullable = false)
 |-- Indirim_x_Haftasonu: double (nullable = false)
 |-- ulke_Index: double (nullable = false)
 |-- magazakod_Index: double (nullable = false)
 |-- merchmarkayasgrupkod_Inde

In [10]:
from pyspark.ml.feature import VectorAssembler

print("Makine Öğrenmesi Ön Hazırlığı: Vektör Pres")


bagimsiz_degiskenler = [
    'urunklasmankod', 
    'satisadet_lfl_gy',      
    'indirimorani', 
    'gunlukminimumsicaklik', 
    'gunlukortalamasicaklik', 
    'gunlukmaksimumsicaklik', 
    'ozelgunflag', 
    'gunsonutoplamstok', 
    'gunsonureyonstok', 
    'Haftanin_Gunu', 
    'Yilin_Ayi',              # Sezonsallık
    'Yilin_Haftasi',          # Sezonsallık
    'Acik_Kalma_Suresi_Saat', 
    'ulke_Index', 
    'magazakod_Index', 
    'merchmarkayasgrupkod_Index', 
    'merchaltgrupkod_Index', 
    'depoyerlesimtip_Index',
    'Haftasonu_Mu',
    'Indirim_x_Haftasonu'
]


assembler = VectorAssembler(
    inputCols=bagimsiz_degiskenler,
    outputCol="features", 
    handleInvalid="skip"  
)

# Veriyi dönüştür
df_vektor = assembler.transform(df_ml_hazir)
train_df, test_df = df_vektor.randomSplit([0.8, 0.2], seed=42)

print("-" * 50)
print(f"Eğitim Verisi (Train) Satır Sayısı: {train_df.count()}")
print(f"Test Verisi (Test) Satır Sayısı:   {test_df.count()}")
print("-" * 50)


train_df.select("features", "satisadet").show(5, truncate=False)

Makine Öğrenmesi Ön Hazırlığı: Vektör Pres
--------------------------------------------------


Eğitim Verisi (Train) Satır Sayısı: 860819


Test Verisi (Test) Satır Sayısı:   216561
--------------------------------------------------
+----------------------------------------------------------------------------------------+---------+
|features                                                                                |satisadet|
+----------------------------------------------------------------------------------------+---------+
|[1023.0,0.0,0.0,-1.0,1.3,4.0,0.0,2.0,2.0,4.0,2.0,6.0,12.12,4.0,0.0,4.0,5.0,1.0,0.0,0.0] |0        |
|[1023.0,0.0,0.0,-1.0,3.29,7.0,0.0,2.0,2.0,1.0,2.0,6.0,12.37,4.0,0.0,4.0,5.0,1.0,1.0,0.0]|0        |
|[1023.0,0.0,0.0,0.0,3.89,7.0,1.0,0.0,0.0,4.0,2.0,5.0,12.13,4.0,0.0,4.0,5.0,1.0,0.0,0.0] |0        |
|[1023.0,0.0,0.0,0.0,4.5,8.0,1.0,0.0,0.0,3.0,2.0,7.0,12.1,4.0,0.0,4.0,5.0,1.0,0.0,0.0]   |0        |
|(20,[0,4,5,9,10,11,12,13,15,16,17],[1023.0,10.6,13.0,6.0,1.0,1.0,12.58,4.0,4.0,5.0,1.0])|0        |
+----------------------------------------------------------------------------------------+---------

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor as cbr
from sklearn.linear_model import LinearRegression

kullanilacak_sutunlar = bagimsiz_degiskenler + ["satisadet"]

pdf_train = train_df.select(kullanilacak_sutunlar).toPandas()
pdf_test = test_df.select(kullanilacak_sutunlar).toPandas()

# X (Özellikler) ve Y (Hedef) olarak ayırıyoruz
X_train = pdf_train.drop("satisadet", axis=1)
y_train = pdf_train["satisadet"]

X_test = pdf_test.drop("satisadet", axis=1)
y_test = pdf_test["satisadet"]

print("Veri yerel belleğe alındı.")

# Baseline model Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)
print("Baseline Model: Linear Regression")
print(f"Linear Regression RMSE : {rmse_lr:.2f}")
print(f"Linear Regression R-Squared : {r2_lr:.4f}")

# Kfold Cross-Validation ile CatBoost ve LGBM karşılaştırması
kf = KFold(n_splits=5, shuffle=True, random_state=42)
catboost_rmse_scores = []
catboost_r2_scores = []
lgbm_rmse_scores = []
lgbm_r2_scores = []
stacked_rmse_scores = []
stacked_r2_scores = []
for train_index, val_index in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    # CatBoost Model
    cat_model = cbr(
        iterations=150,
        learning_rate=0.1,
        depth=16,
        random_seed=42,
        verbose=0
    )
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=10)
    y_pred_cat = cat_model.predict(X_test)
    rmse_cat = np.sqrt(mean_squared_error(y_test, y_pred_cat))
    r_squared_cat = r2_score(y_test, y_pred_cat)
    catboost_rmse_scores.append(rmse_cat)
    catboost_r2_scores.append(r_squared_cat)

    # LightGBM Model
    lgbm_model = lgb.LGBMRegressor(
        n_estimators=150,
        learning_rate=0.1,
        max_depth=16,
        random_state=42,
        n_jobs=-1
    )
    lgbm_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],callbacks=[lgb.early_stopping(10), lgb.log_evaluation(0)])
    y_pred_lgbm = lgbm_model.predict(X_test)
    rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
    r_squared_lgbm = r2_score(y_test, y_pred_lgbm)
    lgbm_rmse_scores.append(rmse_lgbm)
    lgbm_r2_scores.append(r_squared_lgbm)
    
    # Stacked Model (CatBoost + LGBM)
    stacked_pred = (y_pred_cat + y_pred_lgbm) / 2
    rmse_stacked = np.sqrt(mean_squared_error(y_test, stacked_pred))
    stacked_rmse_scores.append(rmse_stacked)
    r_squared_stacked = r2_score(y_test, stacked_pred)
    stacked_r2_scores.append(r_squared_stacked)

print(f"CatBoost Ortalama RMSE: {np.mean(catboost_rmse_scores):.2f} ")
print(f"LightGBM Ortalama RMSE: {np.mean(lgbm_rmse_scores):.2f} ")
print(f"Stacked Model Ortalama RMSE: {np.mean(stacked_rmse_scores):.2f} ")
print(f"CatBoost Ortalama R-Squared: {np.mean(catboost_r2_scores):.4f} ")
print(f"LightGBM Ortalama R-Squared: {np.mean(lgbm_r2_scores):.4f} ")
print(f"Stacked Model Ortalama R-Squared: {np.mean(stacked_r2_scores):.4f} ")

In [ ]:
import joblib
joblib.dump(lgbm_model, "stacked_model.pkl")
print("Model diske kaydedildi: 'stacked_model.pkl'")

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı. Tüm işlemler tamamlandı!")